# Kaggle runner for the AI631 project

1. Upload `25280058_AI631_Project_code_fixed.zip` as a Kaggle **Dataset** (Kaggle unzips it).
2. In this notebook: *Add Input* -> that dataset. Settings: **CPU**, **Internet on**.
3. Use **Save Version -> Save & Run All (Commit)** so the long noise notebook runs in the background (12 hour limit),
   then download `Project_outputs.zip` from the Output tab.

Each cell below runs one project notebook with `nbconvert`, in place, so the executed notebooks (with outputs) end up in the zip.

In [ ]:
# Qiskit is not preinstalled on Kaggle
!pip install -q "qiskit>=2.1,<3" "qiskit-aer>=0.17" "qiskit-ibm-runtime>=0.40" pylatexenc

In [ ]:
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

# Find the uploaded project (the folder that holds qsvm_rc) and copy it to the writable working folder
source = next(path.parent for path in Path("/kaggle/input").rglob("qsvm_rc") if path.is_dir())
WORK = Path("/kaggle/working/Project")
if not WORK.exists():
    shutil.copytree(source, WORK)
print("Source:", source)
print("Working copy:", WORK)
print("CPU cores:", os.cpu_count())

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__, "| runtime", qiskit_ibm_runtime.__version__)

In [ ]:
def run_notebook(relative_path):
    """Execute one project notebook in place, from its own folder, with no time limit."""
    path = WORK / relative_path
    start = time.time()
    result = subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace",
                             "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=python3",
                             path.name],
                            cwd = path.parent, capture_output = True, text = True)
    minutes = (time.time() - start) / 60
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f"{relative_path} failed after {minutes:.1f} min, see the error above")
    print(f"OK  {relative_path}  ({minutes:.1f} min)")

In [ ]:
run_notebook("00_setup/00_environment_and_sanity_checks.ipynb")

In [ ]:
run_notebook("02_classical/01_classical_baselines.ipynb")

In [ ]:
run_notebook("03_quantum/01_qsvm_main_learning_curves.ipynb")

In [ ]:
run_notebook("04_ablations/01_encoding_ablations.ipynb")

In [ ]:
run_notebook("04_ablations/02_training_ablations.ipynb")

In [ ]:
run_notebook("04_ablations/03_other_sections.ipynb")

In [ ]:
# The long one (shots, depolarizing, readout, fake device). Uses every CPU core.
run_notebook("05_noise_hardware/01_noise_study_aer.ipynb")

In [ ]:
run_notebook("06_analysis/01_final_tables_and_falsifier.ipynb")

In [ ]:
# Zip notebooks, helper code and results (the large dataset CSVs are left out, you already have them)
archive = Path("/kaggle/working/Project_outputs.zip")
skip_suffixes = {".pyc"}
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for file in WORK.rglob("*"):
        if file.is_dir() or file.suffix in skip_suffixes or ".ipynb_checkpoints" in file.parts:
            continue
        if file.suffix == ".csv" and file.parent in (WORK, WORK / "extra_sections_data"):
            continue
        bundle.write(file, Path("Project") / file.relative_to(WORK))
print(f"{archive} ({archive.stat().st_size / 1e6:.1f} MB)")